# Étape 1 — Exploration des données (version Jupyter)

Ce notebook est l'équivalent de `01_exploration.py`. Il ne modifie aucun script
du projet : il lit les huit CSV bruts, construit le même échantillon stratifié
de 5 % et écrit ses figures et ses tableaux dans `reports/`.

Règles conservées :

- graine `RANDOM_STATE = 42` partout ;
- familles de moins de 100 lignes gardées entièrement ;
- fichiers bruts laissés intacts ;
- sorties dans `reports/figures/` et `reports/tables/`.

Déroulement : dossier de travail → chargement → libellés → échantillon →
qualité → distributions → corrélations.

In [ ]:
import os
import sys
from pathlib import Path

# Jupyter ouvre le notebook depuis notebooks/ : on remonte jusqu'à config.py
dossier = Path.cwd()
while dossier != dossier.parent and not (dossier / "config.py").exists():
    dossier = dossier.parent
os.chdir(dossier)
if str(dossier) not in sys.path:
    sys.path.insert(0, str(dossier))
print("Dossier du projet :", dossier)

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from config import (RANDOM_STATE, SAMPLE_FRACTION, RARE_MAX, SAMPLE_FILE,
                    RAW_DIR, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR,
                    FAMILIES, FAMILY_ORDER, KEY_FEATURES)

sns.set_theme(style="whitegrid")
print("Graine utilisee :", RANDOM_STATE)

## 1. Chargement des huit fichiers CSV

Les colonnes des fichiers officiels commencent par un espace : on les nettoie.
On ajoute `source_file` et `jour` pour savoir d'où vient chaque flux.

In [ ]:
def charger_csv():
    dossier = RAW_DIR / "MachineLearningCVE"
    if not dossier.exists():
        dossier = RAW_DIR

    fichiers = sorted(dossier.glob("*.csv"))
    if not fichiers:
        print("Place les 8 CSV dans data/raw ou data/raw/MachineLearningCVE.")
        raise SystemExit(1)

    frames = []
    for fichier in fichiers:
        tmp = pd.read_csv(fichier, low_memory=False)
        tmp.columns = [col.strip() for col in tmp.columns]
        tmp["source_file"] = fichier.name
        tmp["jour"] = fichier.name.split("-")[0]
        frames.append(tmp)
        print("lu", fichier.name, len(tmp), "lignes")

    return pd.concat(frames, ignore_index=True)


def echantillonner(df):
    morceaux = []
    for famille, groupe in df.groupby("Family"):
        n = len(groupe) if len(groupe) <= RARE_MAX else round(len(groupe) * SAMPLE_FRACTION)
        morceaux.append(groupe.sample(n, random_state=RANDOM_STATE))
    return pd.concat(morceaux).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)


def sauvegarder_table(table, nom):
    table.to_csv(TABLES_DIR / nom, index=False)


def sauvegarder_figure(fig, nom):
    fig.savefig(FIGURES_DIR / nom, dpi=120, bbox_inches="tight")
    plt.close(fig)

In [ ]:
df = charger_csv()
print("\nBrut :", len(df), "lignes et", df.shape[1], "colonnes")

## 2. Nettoyage des libellés

Certains CSV contiennent un caractère bizarre à la place d'un espace. On
remplace les signes isolés, puis on regroupe les 15 libellés officiels en
familles grâce au dictionnaire `FAMILIES` de `config.py`.

In [ ]:
df["Label"] = df["Label"].astype(str).str.strip()
df["Label"] = df["Label"].str.replace(r"[^\w\s-]+", " ", regex=True)
df["Label"] = df["Label"].str.replace(r"\s+", " ", regex=True)

lookup = {nom.lower(): famille for nom, famille in FAMILIES.items()}
df["Family"] = df["Label"].str.lower().map(lookup).fillna("AUTRE")
inconnus = sorted(df.loc[df["Family"] == "AUTRE", "Label"].unique())
print("Libellés non reconnus :", inconnus)

print("\nFamilles dans le jeu complet :")
print(df["Family"].value_counts().to_string())

## 3. Échantillon stratifié à 5 %

On prend 5 % des lignes de chaque famille, avec la graine 42. Les familles qui
ont 100 lignes ou moins sont gardées entièrement pour ne pas les perdre.

In [ ]:
df = echantillonner(df)
df.to_csv(PROCESSED_DIR / SAMPLE_FILE, index=False)
print("\nÉchantillon :", len(df), "lignes")
print("Mémoire (Mo) :", round(df.memory_usage(deep=True).sum() / 1e6, 1))
print("Types :", df.dtypes.value_counts().to_dict())

## 4. Distribution des classes

`BENIGN` domine largement : l'échelle logarithmique est nécessaire pour voir
les familles minoritaires. Cette observation explique pourquoi on utilisera
le F1 macro et non l'accuracy.

In [ ]:
ordre = [famille for famille in FAMILY_ORDER if famille in df["Family"].unique()]
familles = df["Family"].value_counts().reindex(ordre)
distribution = pd.DataFrame({"famille": familles.index,
                             "effectif": familles.values,
                             "pourcent": (100 * familles.values / len(df)).round(2)})
sauvegarder_table(distribution, "distribution_classes.csv")
distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(distribution["famille"], distribution["effectif"], color="steelblue")
ax.set_yscale("log")
ax.set_title("Nombre de flux par famille (échelle log)")
ax.set_ylabel("Nombre de flux")
ax.tick_params(axis="x", rotation=35)
plt.show()
sauvegarder_figure(fig, "distribution_classes.png")

## 5. Qualité des données

On compte quatre choses : valeurs manquantes, valeurs infinies, colonnes
constantes et doublons. Pour les doublons, on ignore `Family`, `source_file`
et `jour` qui ont été ajoutés par le notebook.

In [ ]:
manquants = df.isna().sum()
manquants = manquants[manquants > 0].sort_values(ascending=False)
table_manquants = pd.DataFrame({"colonne": manquants.index,
                                "nb_manquant": manquants.values,
                                "pourcent": (100 * manquants.values / len(df)).round(3)})
sauvegarder_table(table_manquants, "valeurs_manquantes.csv")

colonnes_num = list(df.select_dtypes(include="number").columns)
nb_infinis = {}
for col in colonnes_num:
    nb = int(np.isinf(df[col].to_numpy(dtype=float)).sum())
    if nb > 0:
        nb_infinis[col] = nb
table_infinis = pd.DataFrame({"colonne": list(nb_infinis),
                              "nb_infini": list(nb_infinis.values())})
sauvegarder_table(table_infinis, "valeurs_infinies.csv")

uniques = df.nunique(dropna=False)
constantes = uniques[uniques <= 1]
sauvegarder_table(pd.DataFrame({"colonne": constantes.index}), "colonnes_constantes.csv")

colonnes_features = [c for c in colonnes_num]
doublons_exacts = int(df.duplicated(colonnes_features + ["Label"]).sum())
doublons_features = int(df.duplicated(colonnes_features).sum())
table_doublons = pd.DataFrame({
    "type": ["lignes identiques (features + label)", "mêmes features, label ignoré"],
    "nombre": [doublons_exacts, doublons_features],
    "pourcent": [round(100 * doublons_exacts / len(df), 2),
                 round(100 * doublons_features / len(df), 2)]
})
sauvegarder_table(table_doublons, "doublons.csv")

print("Qualité :")
print("  valeurs manquantes :", int(manquants.sum()), "cellules")
print("  valeurs infinies   :", sum(nb_infinis.values()), "cellules")
print("  colonnes constantes:", len(constantes))
print("  doublons           :", doublons_exacts, "(", table_doublons.loc[0, "pourcent"], "%)")
table_doublons

In [ ]:
if len(table_manquants) > 0:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(table_manquants["colonne"], table_manquants["pourcent"], color="indianred")
    ax.set_title("Colonnes avec des valeurs manquantes")
    ax.set_ylabel("Pourcentage")
    ax.tick_params(axis="x", rotation=35)
    plt.show()
    sauvegarder_figure(fig, "valeurs_manquantes.png")

## 6. Distributions par famille et corrélations

On garde au plus 1 000 lignes par famille pour des boîtes lisibles. L'échelle
`symlog` permet de voir à la fois les petites et les grandes valeurs. Ensuite
on repère les paires de variables corrélées à plus de 0,95.

In [ ]:
plot_df = df.groupby("Family", group_keys=False).head(1000)

for feature in KEY_FEATURES:
    if feature not in df.columns:
        continue
    fig, ax = plt.subplots(figsize=(11, 4))
    sns.boxplot(data=plot_df, x="Family", y=feature, order=ordre,
                showfliers=False, ax=ax, color="lightsteelblue")
    ax.set_yscale("symlog", linthresh=0.1)
    ax.set_title(feature + " par famille")
    ax.tick_params(axis="x", rotation=35)
    plt.show()
    sauvegarder_figure(fig, "box_" + feature.lower().replace(" ", "_").replace("/", "_") + ".png")

In [ ]:
for feature in ["Flow Duration", "Packet Length Mean"]:
    fig, ax = plt.subplots(figsize=(8, 4))
    for famille in ordre[:4]:
        valeurs = plot_df.loc[plot_df["Family"] == famille, feature]
        valeurs = valeurs[valeurs > 0]
        if len(valeurs) > 0:
            ax.hist(valeurs, bins=np.logspace(np.log10(valeurs.min()),
                    np.log10(valeurs.max()), 30), alpha=0.5, label=famille)
    ax.set_xscale("log")
    ax.set_title("Histogramme log : " + feature)
    ax.set_xlabel(feature)
    ax.legend(fontsize=8)
    plt.show()
    sauvegarder_figure(fig, "hist_" + feature.lower().replace(" ", "_") + ".png")

In [ ]:
corr = df[colonnes_num].corr()
fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(corr, cmap="coolwarm", vmin=-1, vmax=1, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Corrélations entre variables numériques")
ax.tick_params(axis="x", rotation=90, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
plt.show()
sauvegarder_figure(fig, "correlations.png")

masque = np.triu(np.ones(corr.shape, dtype=bool), k=1)
paires = corr.where(masque).stack().rename("correlation").reset_index()
paires.columns = ["variable_1", "variable_2", "correlation"]
paires = paires[paires["correlation"].abs() > 0.95]
paires = paires.reindex(paires["correlation"].abs().sort_values(ascending=False).index)
sauvegarder_table(paires, "correlations_fortes.csv")
print("Paires de variables corrélées à plus de 0,95 :", len(paires))
paires.head(10)

## Résumé de l'étape 1 (version Jupyter)

Résultats identiques à ceux du script `01_exploration.py` :

- 2 830 743 lignes brutes, échantillon de 141 581 lignes (5 % stratifié) ;
- 80,28 % de trafic `BENIGN` ;
- 3,92 % de doublons, 162 valeurs infinies, 8 colonnes constantes ;
- 48 paires de variables corrélées à plus de 0,95 ;
- 10 figures et 6 tableaux écrits dans `reports/`.

Le jeu de test n'existe pas encore à ce stade : il sera séparé à l'étape 2 et
ne sera utilisé qu'à l'étape 5.

In [ ]:
print("\nFin de l'étape 1 (version Jupyter). Figures et tableaux écrits dans reports/.")